In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from uncertainties import ufloat as uf
from uncertainties import unumpy as unp
from pathlib import Path

# path config
downloads_dir = Path.home() / 'Downloads'
# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistota typu B u cteni z nejmensiho dilku
def unc_B_cteni(a):
    '''
    Nejistota typu B pro napr pravitko
    data : float
        nejmensi dilek
    return : float
        nejistota typu B
    '''
    u_B = a/(3**0.5)
    return u_B

# nejistota typu A 
def unc_A(data):
    '''
    Výpočet nejistoty typu A
    data : list nebo numpy array
         měření
    return : float
        nejistota typu A
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)
    
    return u_A

In [ ]:
# nacteni google tabulek
sheet_id = '1WguDcMu3y95ozOn7a2cBLHqNAV9YbfoBmDYihw6zr6o'
gid = '0'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_hranol = pd.read_csv(url)
df_hranol.dropna()

In [ ]:
# lamavy uhel

psi1_df = df_hranol['ψ₁ – stupně'] + df_hranol['ψ₁ – minuty']/60 + df_hranol['ψ₁ – vteřiny']/3600
psi1 = psi1_df.to_numpy(dtype=float)

psi2_df = df_hranol['ψ₂ – stupně'] + df_hranol['ψ₂ – minuty']/60 + df_hranol['ψ₂ – vteřiny']/3600
psi2 = psi2_df.to_numpy(dtype=float)

omega = 180 - (psi1 - psi2)

In [ ]:
# nacteni google tabulek
sheet_id = '1WguDcMu3y95ozOn7a2cBLHqNAV9YbfoBmDYihw6zr6o'
gid = '37383797'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_deviace = pd.read_csv(url)
df_deviace

In [ ]:
# deviace

phi1_df = df_deviace['φ₁ – stupně'] + df_deviace['φ₁ – minuty']/60 + df_deviace['φ₁ – vteřiny']/3600
phi1 = phi1_df.to_numpy(dtype=float)

phi2_df = df_deviace['φ₂ – stupně'] + df_deviace['φ₂ – minuty']/60 + df_deviace['φ₂ – vteřiny']/3600
phi2 = phi2_df.to_numpy(dtype=float)

delta_min = (phi1 - phi2)/2
# index lomu

n = np.sin((delta_min+omega)/2)/np.sin(omega/2)

In [ ]:
# cauchy fit
vln_delka = df_deviace['λ [nm]']
vln_delka_lispace = np.linspace(350*1e-9, 750*1e-9, 1000)

def cauchy(vln_delka, A, B, C):
    x = 1/(vln_delka**2)
    return A + B*x + C*x**2

p0 = [1, 1, 1]
popt, pcov = curve_fit(cauchy, vln_delka, n, p0)
A, B, C = popt
unc_A, unc_B, unc_C = np.sqrt(np.diag(pcov))

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_ylabel(fr"Odrazivost", fontsize=20)
ax.set_xlabel(fr"$\varphi_0\,[°]$", fontsize=20)
ax.scatter(vln_delka, n, label='Naměřenné hodnoty')
ax.plot(vln_delka_lispace, cauchy(vln_delka_lispace, *popt), label='Fit')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
plt.savefig(downloads_dir / 'cauchy.pdf', dpi=300, bbox_inches='tight')